# DATA ANALYTICS ML CHALLENGE / 2026-2
## Laboratorio 1: modelos supervisados y métricas | Semana 12

**Estudiante:** Sara Valentina Guauque Zarrate  
**Asignatura:** Data Analytics, Ingeniería de Sistemas y Computación  
**Universidad:** Universidad Central  
**Profesor:** Elias Buitrago Bolivar  
**Caso:** Brazilian E-Commerce Public Dataset by Olist

**Pregunta de negocio:** con los datos conocidos al confirmar una compra, ¿podemos identificar los pedidos con mayor riesgo de entregarse después de la fecha estimada?

El trabajo sigue la plantilla oficial del docente: Carga, Preparación, Modelado, Evaluación y Conclusión de Negocio. Los datos originales y el notebook fuente se conservan en su planteamiento metodologico. No se usan las fechas posteriores a la compra como variables predictoras.

> **Identificador individual confirmado: ID 17.** Semilla fija = 1017. La muestra de 35.000 pedidos y las particiones deben mantenerse para las semanas 12 a 16.


## 1. Carga de datos

Una observación del modelo corresponde a un pedido, aunque tenga varios productos y pagos. Se integrarán siete tablas del conjunto público de Olist.

In [ ]:
import os
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.base import clone
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, mean_absolute_error)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 130)


In [ ]:
# Datos originales de la revision fija indicada por el profesor.
# El cambio opcional OLIST_DATA_DIR permite probar sin conexion si los CSV
# originales ya existen localmente. En Colab se usa el origen publico fijo.
ORIGEN_PUBLICO = ('https://raw.githubusercontent.com/spdrio/'
                  'Brazilian-E-Commerce-Public-Dataset-by-Olist/'
                  'fa0cc6d1aff713d3c8b9a1e633b56889881848bf/files')
ORIGEN = os.environ.get('OLIST_DATA_DIR', ORIGEN_PUBLICO).rstrip('/')

try:
    pedidos = pd.read_csv(f'{ORIGEN}/olist_orders_dataset.csv')
    items = pd.read_csv(f'{ORIGEN}/olist_order_items_dataset.csv')
    productos = pd.read_csv(f'{ORIGEN}/olist_products_dataset.csv')
    clientes = pd.read_csv(f'{ORIGEN}/olist_customers_dataset.csv')
    vendedores = pd.read_csv(f'{ORIGEN}/olist_sellers_dataset.csv')
    pagos = pd.read_csv(f'{ORIGEN}/olist_order_payments_dataset.csv')
    categorias = pd.read_csv(f'{ORIGEN}/product_category_name_translation.csv')
except Exception as exc:
    raise RuntimeError(
        'No se pudieron leer los siete CSV de Olist. Compruebe la conexion ' 
        'a internet en Colab o configure OLIST_DATA_DIR con los archivos originales.'
    ) from exc

for nombre, tabla in [('Pedidos', pedidos), ('Items', items),
                      ('Productos', productos), ('Clientes', clientes),
                      ('Vendedores', vendedores), ('Pagos', pagos),
                      ('Categorias', categorias)]:
    print(f'{nombre:12s}: {tabla.shape[0]:>7,} filas | {tabla.shape[1]} columnas')


## 2. Preparación de variables

Un pedido puede contener varios ítems y pagos. Primero se resumen estas tablas por `order_id`, para evitar contar un mismo pedido varias veces. Las variables de destino se construyen después y no se incluyen entre los predictores.

In [ ]:
# Información de los productos y vendedores incluidos en cada pedido.
items = items.merge(productos, on='product_id', how='left', validate='many_to_one')
items = items.merge(vendedores, on='seller_id', how='left', validate='many_to_one')
items = items.merge(categorias, on='product_category_name',
                    how='left', validate='many_to_one')

items['volumen_cm3'] = (items['product_length_cm'] * items['product_height_cm']
                       * items['product_width_cm'])
items['categoria'] = items['product_category_name_english'].fillna(
    items['product_category_name']).fillna('sin_categoria')

# Resumen de productos: una fila por pedido.
resumen_items = items.groupby('order_id', as_index=False).agg(
    valor_productos=('price', 'sum'),
    valor_flete=('freight_value', 'sum'),
    precio_maximo=('price', 'max'),
    cantidad_items=('order_item_id', 'count'),
    cantidad_productos=('product_id', 'nunique'),
    cantidad_vendedores=('seller_id', 'nunique'),
    peso_total_g=('product_weight_g', 'sum'),
    peso_maximo_g=('product_weight_g', 'max'),
    volumen_total_cm3=('volumen_cm3', 'sum'),
    fotos_promedio=('product_photos_qty', 'mean'),
    descripcion_promedio=('product_description_lenght', 'mean'),
    nombre_promedio=('product_name_lenght', 'mean'),
)

# Categoría y vendedor del producto de mayor precio del pedido.
principal = (items.sort_values(['order_id', 'price', 'order_item_id'],
                                ascending=[True, False, True])
             .drop_duplicates('order_id'))
principal = principal[['order_id', 'categoria', 'seller_state', 'seller_city']]
principal = principal.rename(columns={
    'categoria': 'categoria_principal',
    'seller_state': 'estado_vendedor',
    'seller_city': 'ciudad_vendedor'
})

In [ ]:
# Los pagos también pueden aparecer en varias filas por pedido.
resumen_pagos = pagos.groupby('order_id', as_index=False).agg(
    numero_pagos=('payment_sequential', 'count'),
    cuotas_maximas=('payment_installments', 'max'),
    valor_pagado=('payment_value', 'sum'),
)

pago_principal = (pagos.sort_values(['order_id', 'payment_value'],
                                    ascending=[True, False])
                  .drop_duplicates('order_id'))
pago_principal = pago_principal[['order_id', 'payment_type']].rename(
    columns={'payment_type': 'tipo_pago_principal'})

# Integración de las tablas: una fila por pedido.
data = pedidos.merge(clientes[['customer_id', 'customer_state', 'customer_city']],
                     on='customer_id', how='inner', validate='one_to_one')
data = data.merge(resumen_items, on='order_id', how='inner', validate='one_to_one')
data = data.merge(principal, on='order_id', how='left', validate='one_to_one')
data = data.merge(resumen_pagos, on='order_id', how='left', validate='one_to_one')
data = data.merge(pago_principal, on='order_id', how='left', validate='one_to_one')

assert data['order_id'].is_unique, 'Hay pedidos duplicados después de las uniones.'
print('Pedidos integrados:', len(data))

In [ ]:
# Se consideran pedidos entregados con fechas completas y coherentes.
fechas = ['order_purchase_timestamp', 'order_estimated_delivery_date',
          'order_delivered_customer_date']
for columna in fechas:
    data[columna] = pd.to_datetime(data[columna], errors='coerce')

data = data.loc[data['order_status'] == 'delivered'].dropna(subset=fechas).copy()
compra = data['order_purchase_timestamp'].dt.normalize()
prevista = data['order_estimated_delivery_date'].dt.normalize()
real = data['order_delivered_customer_date'].dt.normalize()

data['dias_prometidos'] = (prevista - compra).dt.days
data['dias_entrega'] = (real - compra).dt.days
data['is_late'] = (real > prevista).astype(int)
data['mes_compra'] = compra.dt.month
data['dia_semana_compra'] = compra.dt.dayofweek
data['dia_mes_compra'] = compra.dt.day
data['hora_compra'] = data['order_purchase_timestamp'].dt.hour
data['compra_fin_semana'] = (data['dia_semana_compra'] >= 5).astype(int)

data['vendedor_mismo_estado'] = (
    data['customer_state'] == data['estado_vendedor']).astype(int)
data['vendedor_misma_ciudad'] = (
    data['vendedor_mismo_estado'].eq(1) &
    data['customer_city'].str.lower().eq(data['ciudad_vendedor'].str.lower())
).astype(int)
data['proporcion_flete'] = data['valor_flete'] / (
    data['valor_productos'] + data['valor_flete']).replace(0, np.nan)

data = data.loc[(data['dias_prometidos'] >= 0) &
                (data['dias_entrega'] >= 0)].copy()
data = data.sort_values('order_id').reset_index(drop=True)

print('Pedidos disponibles:', len(data))
print('Entregas tardías:', round(100 * data['is_late'].mean(), 2), '%')

### Variables del modelo

Se utilizarán datos que razonablemente están disponibles al cierre de la compra: plazo prometido, calendario, contenido del pedido, dimensiones de los productos, ubicación y medio de pago. Las fechas reales de envío y entrega, y las reseñas posteriores, quedan excluidas de las variables predictoras.

La lista de variables y sus tablas de origen se encuentra en el diccionario técnico anexo.

In [ ]:
columnas_numericas = [
    'dias_prometidos', 'mes_compra', 'dia_semana_compra',
    'dia_mes_compra', 'hora_compra', 'compra_fin_semana',
    'valor_productos', 'valor_flete', 'precio_maximo',
    'cantidad_items', 'cantidad_productos', 'cantidad_vendedores',
    'peso_total_g', 'peso_maximo_g', 'volumen_total_cm3',
    'fotos_promedio', 'descripcion_promedio', 'nombre_promedio',
    'numero_pagos', 'cuotas_maximas', 'valor_pagado',
    'vendedor_mismo_estado', 'vendedor_misma_ciudad', 'proporcion_flete'
]
columnas_categoricas = [
    'customer_state', 'estado_vendedor',
    'categoria_principal', 'tipo_pago_principal'
]
columnas = columnas_numericas + columnas_categoricas

assert data['order_id'].is_unique
assert len(data) >= 35000
print('Variables predictoras:', len(columnas))
print('Numéricas:', len(columnas_numericas), '| Categóricas:', len(columnas_categoricas))

### Muestra individual y particiones

**Estudiante: Sara Valentina Guauque Zarrate | ID 17 | semilla 1017.**

La plantilla del profesor toma 35.000 pedidos mediante muestreo estratificado y la semilla `1000 + ID_ESTUDIANTE`. La division se mantiene en 24.500 registros para entrenamiento (70 %), 5.250 para validacion (15 %) y 5.250 para prueba (15 %). El conjunto de prueba se reserva para la semana 16 y no se utiliza para seleccionar modelos en S12.


In [ ]:
# ID OFICIAL CONFIRMADO de Sara Valentina Guauque Zarrate.
# No modificar: todas las semanas deben conservar este identificador.
ID_ESTUDIANTE = 17
if not isinstance(ID_ESTUDIANTE, int) or isinstance(ID_ESTUDIANTE, bool) or not 1 <= ID_ESTUDIANTE <= 30:
    raise ValueError('Antes de ejecutar, escriba su ID OFICIAL (entero de 1 a 30) en ID_ESTUDIANTE.')

semilla = 1000 + ID_ESTUDIANTE
identificador = f'{ID_ESTUDIANTE:02d}'

muestra, _ = train_test_split(data, train_size=35000,
                             random_state=semilla, stratify=data['is_late'])
muestra = muestra.sort_values('order_id').reset_index(drop=True)

train, temporal = train_test_split(muestra, test_size=0.30,
                                  random_state=semilla, stratify=muestra['is_late'])
valid, test = train_test_split(temporal, test_size=0.50,
                               random_state=semilla, stratify=temporal['is_late'])

X_train, y_train = train[columnas], train['is_late']
X_valid, y_valid = valid[columnas], valid['is_late']

huella_muestra = hashlib.sha256('|'.join(muestra['order_id']).encode()).hexdigest()[:12]
print('Estudiante ID:', identificador, '| Semilla:', semilla)
print('Muestra:', len(muestra), '| Train:', len(train),
      '| Valid:', len(valid), '| Test:', len(test))
print('Huella de muestra:', huella_muestra)


## 3. Modelo de referencia y clasificación

Las variables numéricas se imputan y escalan; las categóricas se transforman con *One-Hot Encoding*. Estas transformaciones se ajustan únicamente con los datos de entrenamiento, dentro del `Pipeline`.

In [ ]:
preprocesamiento = ColumnTransformer([
    ('numericas', make_pipeline(SimpleImputer(strategy='median'),
                               StandardScaler()), columnas_numericas),
    ('categoricas', make_pipeline(SimpleImputer(strategy='most_frequent'),
                                 OneHotEncoder(handle_unknown='ignore')),
     columnas_categoricas)
])

# Referencia: elige siempre la clase mayoritaria (normalmente, puntual).
referencia = DummyClassifier(strategy='most_frequent')
referencia.fit(X_train, y_train)
pred_ref = referencia.predict(X_valid)
accuracy_ref = accuracy_score(y_valid, pred_ref)
recall_ref = recall_score(y_valid, pred_ref, zero_division=0)
f1_ref = f1_score(y_valid, pred_ref, zero_division=0)
print(f'Referencia | Accuracy: {accuracy_ref:.4f} | Recall: {recall_ref:.4f} | F1: {f1_ref:.4f}')
print('Prevalencia tardias (validacion):', f'{y_valid.mean():.2%}')


### Actividad 1. Entrenamiento y selección de ponderación de clases

Entreno **dos regresiones logisticas** con la misma preparación, el mismo entrenamiento y la misma validación: `class_weight=None` y `class_weight='balanced'`. Comparo ambos modelos por F1 (y Recall como desempate), porque la entrega tardía es la clase minoritaria y una Accuracy alta por si sola puede ser enganosa. La selección se realiza **solo en validación**, sin tocar `test`.

Conservo las variables exigidas por la plantilla: `peso_clases`, `modelo`, `pred`.


In [ ]:
# Actividad 1: entrenamiento reproducible y comparacion justa sobre la MISMA muestra.
candidatos = {}
filas_comparacion = []
for opcion in (None, 'balanced'):
    nombre = 'sin_ponderacion' if opcion is None else 'balanced'
    estimador = make_pipeline(
        clone(preprocesamiento),
        LogisticRegression(class_weight=opcion, max_iter=1000,
                           random_state=semilla)
    )
    estimador.fit(X_train, y_train)
    pred_cand = estimador.predict(X_valid)
    proba_cand = estimador.predict_proba(X_valid)[:, 1]
    tn_c, fp_c, fn_c, tp_c = confusion_matrix(y_valid, pred_cand, labels=[0, 1]).ravel()
    fila = {
        'configuracion': nombre,
        'accuracy': accuracy_score(y_valid, pred_cand),
        'precision': precision_score(y_valid, pred_cand, zero_division=0),
        'recall': recall_score(y_valid, pred_cand, zero_division=0),
        'f1': f1_score(y_valid, pred_cand, zero_division=0),
        'roc_auc': roc_auc_score(y_valid, proba_cand),
        'ap': average_precision_score(y_valid, proba_cand),
        'fp': int(fp_c), 'fn': int(fn_c),
        'costo_brl': int(8 * fp_c + 35 * fn_c),
    }
    candidatos[nombre] = {'modelo': estimador, 'opcion': opcion, 'fila': fila}
    filas_comparacion.append(fila)

comparacion_modelos = pd.DataFrame(filas_comparacion)
seleccion = max(candidatos, key=lambda key: (
    candidatos[key]['fila']['f1'], candidatos[key]['fila']['recall']))
peso_clases = candidatos[seleccion]['opcion']
modelo = candidatos[seleccion]['modelo']
pred = modelo.predict(X_valid)

print('Comparacion de ponderacion en VALIDACION:')
display(comparacion_modelos.round(4))
print(f'Seleccion: {seleccion} | criterio: F1 (desempate Recall) | peso_clases={peso_clases!r}')


## 4. Evaluación

### Actividad 2. Métricas

Calcule Accuracy, Precision, Recall, F1, ROC-AUC y Average Precision (AP) en validación. Para las métricas AUC use las probabilidades que estima el modelo.

In [ ]:
# Actividad 2: seis metricas exigidas y matriz de confusion en validacion.
proba = modelo.predict_proba(X_valid)[:, 1]
medidas = {
    'accuracy': float(accuracy_score(y_valid, pred)),
    'precision': float(precision_score(y_valid, pred, zero_division=0)),
    'recall': float(recall_score(y_valid, pred, zero_division=0)),
    'f1': float(f1_score(y_valid, pred, zero_division=0)),
    'roc_auc': float(roc_auc_score(y_valid, proba)),
    'ap': float(average_precision_score(y_valid, proba)),
}
matriz = confusion_matrix(y_valid, pred, labels=[0, 1])
print('Matriz de confusion [filas: real 0/1; columnas: prediccion 0/1]:')
print(matriz)
print('Metricas del modelo seleccionado:')
for metrica, valor in medidas.items():
    print(f'  {metrica:10s}: {valor:.4f}')

fig, ax = plt.subplots(figsize=(6.6, 4.6))
ax.imshow(matriz, cmap='Blues')
for (fila, columna), cuenta in np.ndenumerate(matriz):
    ax.text(columna, fila, str(cuenta), ha='center', va='center',
            fontsize=14, color='white' if cuenta > matriz.max() / 2 else 'black')
ax.set(xticks=[0, 1], yticks=[0, 1],
       xticklabels=['A tiempo (0)', 'Tarde (1)'],
       yticklabels=['A tiempo (0)', 'Tarde (1)'],
       xlabel='Prediccion', ylabel='Realidad', title='Matriz de confusion | Validacion S12')
fig.tight_layout()
Path('reports/figuras').mkdir(parents=True, exist_ok=True)
fig.savefig(f'reports/figuras/S12_matriz_ID_{identificador}.png', dpi=145, bbox_inches='tight')
plt.show()


### Actividad 3. Errores por segmento

Compare la tasa de error por mes de compra y estado del cliente. Muestre tanto el porcentaje de error como la cantidad de pedidos de cada segmento.

In [ ]:
assert pred is not None, 'Complete primero la Actividad 1.'
analisis = valid.copy()
analisis['pred'] = np.asarray(pred)
analisis['error'] = (analisis['is_late'] != analisis['pred']).astype(int)
analisis['fp'] = ((analisis['is_late'] == 0) & (analisis['pred'] == 1)).astype(int)
analisis['fn'] = ((analisis['is_late'] == 1) & (analisis['pred'] == 0)).astype(int)

# Actividad 3: se muestran tasa Y cantidad para evitar conclusiones sobre bases pequenas.
def tabla_errores(segmento):
    resultado = (analisis.groupby(segmento, dropna=False)
                 .agg(total_pedidos=('order_id', 'size'),
                      pedidos_erroneos=('error', 'sum'),
                      falsos_positivos=('fp', 'sum'),
                      falsos_negativos=('fn', 'sum'),
                      tardios_reales=('is_late', 'sum'))
                 .reset_index())
    resultado['tasa_error_pct'] = (
        100 * resultado['pedidos_erroneos'] / resultado['total_pedidos']).round(2)
    resultado['costo_brl'] = (8 * resultado['falsos_positivos']
                               + 35 * resultado['falsos_negativos'])
    return resultado

analisis_mes = tabla_errores('mes_compra').sort_values('mes_compra')
analisis_estado = tabla_errores('customer_state').sort_values(
    'total_pedidos', ascending=False)
print('Errores por mes de compra:')
display(analisis_mes)
print('Errores por estado del cliente (10 con mayor cantidad de pedidos):')
display(analisis_estado.head(10))
print('Estados con mayor error relativo entre los que tienen al menos 100 pedidos:')
estados_representativos = analisis_estado.query('total_pedidos >= 100')
display(estados_representativos.sort_values('tasa_error_pct', ascending=False).head(5))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].bar(analisis_mes['mes_compra'].astype(str), analisis_mes['tasa_error_pct'])
axes[0].set(title='Error por mes', xlabel='Mes de compra', ylabel='Tasa de error (%)')
axes[1].bar(analisis_estado.head(10)['customer_state'],
            analisis_estado.head(10)['tasa_error_pct'])
axes[1].set(title='Error por estado | top 10 en volumen',
            xlabel='Estado del cliente', ylabel='Tasa de error (%)')
fig.suptitle('Segmentos del conjunto de validacion (n=5.250)')
fig.tight_layout()
fig.savefig(f'reports/figuras/S12_segmentos_ID_{identificador}.png',
            dpi=145, bbox_inches='tight')
plt.show()


### Actividad 4. Costo empresarial

Para el ejercicio, suponga un costo de R$ 8 por falso positivo y R$ 35 por falso negativo. Calcule el costo total de los errores en validación.

In [ ]:
# Actividad 4: costos hipoteticos de negocio indicados por el profesor.
tn, fp, fn, tp = matriz.ravel()
costo_fp_fn = int(8 * fp + 35 * fn)
fp_ref = int(confusion_matrix(y_valid, pred_ref, labels=[0, 1])[0, 1])
fn_ref = int(confusion_matrix(y_valid, pred_ref, labels=[0, 1])[1, 0])
costo_referencia_brl = 8 * fp_ref + 35 * fn_ref
print(f'FP = {fp} x R$ 8 = R$ {8*fp:,}')
print(f'FN = {fn} x R$ 35 = R$ {35*fn:,}')
print(f'Costo total de errores (modelo): R$ {costo_fp_fn:,}')
print(f'Costo de errores (referencia):  R$ {costo_referencia_brl:,}')
if costo_fp_fn > costo_referencia_brl:
    print('Nota: mejorar la deteccion no garantiza minimizar el costo de este supuesto.')
else:
    print('Bajo los costos ilustrativos, el modelo tiene un costo no mayor a la referencia.')


### Autoverificación

Ejecute estas comprobaciones después de completar las actividades.

In [ ]:
assert len(muestra) == 35000, 'La muestra debe tener 35000 pedidos.'
assert (len(train), len(valid), len(test)) == (24500, 5250, 5250), (
    'Particiones esperadas: 24500 / 5250 / 5250. Revise la celda de muestra.')
assert set(train['order_id']).isdisjoint(valid['order_id']), 'Train y valid comparten pedidos.'
assert set(train['order_id']).isdisjoint(test['order_id']), 'Train y test comparten pedidos.'
assert set(valid['order_id']).isdisjoint(test['order_id']), 'Valid y test comparten pedidos.'

assert modelo is not None and pred is not None, 'Complete la Actividad 1.'
assert len(pred) == len(y_valid), 'pred debe tener 5250 predicciones de validación.'
assert set(np.unique(np.asarray(pred))).issubset({0, 1}), 'pred debe contener solo clases 0 y 1.'
assert np.array_equal(np.asarray(pred), modelo.predict(X_valid)), (
    'pred no coincide con modelo.predict(X_valid). Verifique que utilizó X_valid.')

assert all(isinstance(v, (float, int, np.number)) for v in medidas.values()), (
    'Complete las métricas de la Actividad 2.')
assert proba is not None, 'Complete la Actividad 2: calcule las probabilidades.'
assert np.asarray(proba).ndim == 1 and len(proba) == len(y_valid), (
    'proba debe contener una probabilidad por pedido de validación.')
assert np.allclose(proba, modelo.predict_proba(X_valid)[:, 1]), (
    'proba no coincide con modelo.predict_proba(X_valid)[:, 1].')
assert np.isclose(medidas['f1'], f1_score(y_valid, pred, zero_division=0)), (
    "medidas['f1'] no coincide. Use f1_score(y_valid, pred, zero_division=0).")
assert np.isclose(medidas['ap'], average_precision_score(y_valid, proba)), (
    "medidas['ap'] no coincide. Use average_precision_score(y_valid, proba).")
assert np.isclose(medidas['roc_auc'], roc_auc_score(y_valid, proba)), (
    "medidas['roc_auc'] no coincide. Use roc_auc_score(y_valid, proba).")

assert isinstance(costo_fp_fn, (int, float, np.number)), (
    'Complete la Actividad 4: costo_fp_fn debe ser numérico (8*FP + 35*FN).')
_, fp_ver, fn_ver, _ = confusion_matrix(y_valid, pred).ravel()
assert np.isclose(costo_fp_fn, 8 * fp_ver + 35 * fn_ver), (
    'El costo no coincide con 8*FP + 35*FN. Revise la Actividad 4.')
print('Autoverificación superada.')


## 5. Ejemplo guiado de regresión

Con los mismos predictores, estime la duración total de la entrega en días usando `Ridge` y evalúe el error absoluto medio (MAE).

In [ ]:
from sklearn.base import clone

# Cada modelo recibe una copia independiente del preprocesamiento.
modelo_reg = make_pipeline(clone(preprocesamiento), Ridge(solver='lsqr'))
modelo_reg.fit(X_train, train['dias_entrega'])
pred_dias = modelo_reg.predict(X_valid)
mae = mean_absolute_error(valid['dias_entrega'], pred_dias)
print('MAE regresión:', round(mae, 2), 'días')

## 6. Interpretación de negocio

**1. Compare la regresión logística con el modelo de referencia utilizando Accuracy, Recall y F1. Justifique su elección de `class_weight`.**

El modelo de referencia predice la clase más frecuente y por eso puede tener una Accuracy elevada sin identificar entregas tardias. En este laboratorio comparo los modelos con y sin ponderación sobre los mismos 5.250 pedidos de validación y selecciono el de mayor F1, con Recall como desempate. Esta decisión prioriza una mejor detección de la clase minoritaria sobre maximizar solo Accuracy. El efecto exacto de esta elección y sus valores se documentan en la tabla de comparación y en el borrador individual generado al ejecutar el notebook.

*Despues de ejecutar el código con mi ID oficial revisare los valores concretos del reporte para sustentar está conclusion.*


**2. Analice qué tipo de pedidos presenta mayor error según los segmentos estudiados. ¿Cómo se relaciona esto con el costo de falsos positivos y falsos negativos?**

Las tablas anteriores muestran para cada mes y estado del cliente el número de pedidos, los errores y su tasa porcentual. No debe concluirse que un estado es problematico solo por una tasa alta en pocos casos. Tambien diferencio falsos positivos y falsos negativos, porque el costo supuesto es distinto: R$ 8 por falso positivo y R$ 35 por falso negativo. El análisis cuantitativo individual, incluyendo el segmento con mayor error entre los que tienen una base razonable, se genera en `reports/` tras la ejecucion.


**3. Proponga una mejora concreta para la semana 13 y explique como la verificara utilizando la misma validacion.**

En la semana 13 probaré distintos niveles de regularizacion en `LogisticRegression` (`C=0.1`, `1.0` y `10.0`) y el agrupamiento de categorías poco frecuentes antes del One-Hot Encoding. Solo usaré entrenamiento para ajustar cada alternativa y la misma validación para comparar F1, Recall, Average Precision y costo empresarial. Documentaré la variación respecto a esta línea base, sin utilizar los 5.250 pedidos de prueba reservados para la semana 16.


## 7. Registro de resultados

Guarde el resultado del laboratorio y la huella de la muestra. Las indicaciones de entrega están en el documento técnico.

In [ ]:
import sys
import sklearn

# Reporte JSON oficial: conserva los nombres exigidos por el cuaderno base.
reporte = {
    'laboratorio': 'S12',
    'id_estudiante': identificador,
    'semilla': semilla,
    'timestamp_utc': pd.Timestamp.utcnow().isoformat(),
    'python': sys.version.split()[0],
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'sklearn': sklearn.__version__,
    'n_muestra': len(muestra),
    'n_train': len(train),
    'n_valid': len(valid),
    'n_test': len(test),
    'n_variables': len(columnas),
    'columnas_predictoras': columnas,
    'fuente_olist_commit': 'fa0cc6d1aff713d3c8b9a1e633b56889881848bf',
    'tasa_tardias_muestra': float(muestra['is_late'].mean()),
    'huella_muestra': huella_muestra,
    'peso_clases': str(peso_clases),
    'modelo_seleccionado': seleccion,
    'metricas_referencia': {
        'accuracy': float(accuracy_ref),
        'recall': float(recall_ref),
        'f1': float(f1_ref),
    },
    'comparacion_ponderacion': comparacion_modelos.to_dict(orient='records'),
    'matriz_confusion_validacion': matriz.astype(int).tolist(),
    'falsos_positivos': int(fp),
    'falsos_negativos': int(fn),
    'costo_referencia_brl': float(costo_referencia_brl),
    'interpretacion_costos': 'Supuestos didacticos: FP=R$8; FN=R$35',
    'metricas': {nombre: float(valor) for nombre, valor in medidas.items()},
    'costo_errores_brl': float(costo_fp_fn),
    'mae_regresion': float(mae),
}

carpeta_reporte = Path('reports')
carpeta_reporte.mkdir(exist_ok=True)
ruta = carpeta_reporte / f'S12_resultados_ID_{identificador}.json'
ruta.write_text(json.dumps(reporte, indent=2, ensure_ascii=False), encoding='utf-8')
analisis_mes.to_csv(carpeta_reporte / f'S12_errores_mes_ID_{identificador}.csv',
                    index=False, encoding='utf-8-sig')
analisis_estado.to_csv(carpeta_reporte / f'S12_errores_estado_ID_{identificador}.csv',
                       index=False, encoding='utf-8-sig')

# Respuestas numericas de apoyo: se generan SOLO con las metricas del ID real.
mejor_mes = analisis_mes.loc[analisis_mes['tasa_error_pct'].idxmax()]
base_estados = estados_representativos if not estados_representativos.empty else analisis_estado
mejor_estado = base_estados.loc[base_estados['tasa_error_pct'].idxmax()]

respuesta_1 = (
    f'En los {len(valid):,} pedidos de validacion, el clasificador de referencia obtuvo '
    f'Accuracy={accuracy_ref:.3f}, Recall={recall_ref:.3f} y F1={f1_ref:.3f}. '
    f'La regresion logistica seleccionada obtuvo Accuracy={medidas["accuracy"]:.3f}, '
    f'Recall={medidas["recall"]:.3f} y F1={medidas["f1"]:.3f}; '
    f'AP={medidas["ap"]:.3f} y ROC-AUC={medidas["roc_auc"]:.3f}. '
    f'Utilice `class_weight={peso_clases!r}` porque fue la configuracion con mejor F1 '
    f'en la comparacion de validacion (con Recall como desempate). '
    f'Una Accuracy mayor, por si sola, no significa que se detecten retrasos: '
    f'el enfoque es identificar la clase minoritaria sin usar los datos de prueba.'
)
respuesta_2 = (
    f'El mes con mayor porcentaje de errores fue {int(mejor_mes["mes_compra"])} '
    f'con {mejor_mes["tasa_error_pct"]:.2f}% ('
    f'{int(mejor_mes["pedidos_erroneos"])} errores de '
    f'{int(mejor_mes["total_pedidos"])} pedidos). '
    f'Entre los estados con al menos 100 pedidos (o todos, si no hay ninguno), '
    f'el de mayor tasa de error fue {mejor_estado["customer_state"]}: '
    f'{mejor_estado["tasa_error_pct"]:.2f}% con '
    f'{int(mejor_estado["total_pedidos"])} pedidos. '
    f'La matriz de confusion arrojo {int(fp)} falsos positivos y {int(fn)} falsos negativos; '
    f'sus costos ilustrativos fueron R$ {int(8*fp):,} y R$ {int(35*fn):,}, '
    f'para un total de R$ {costo_fp_fn:,} frente a R$ {costo_referencia_brl:,} '
    f'del modelo mayoritario. Un falso negativo cuesta 35 reales y un falso '
    f'positivo 8, pero deben considerarse tanto su cantidad como la tasa de error '
    f'de cada segmento. Las diferencias observadas son descriptivas, no causales.'
)
respuesta_3 = (
    'Para la semana 13 comparare regularizacion de la regresion logistica '
    '(C = 0.1, 1, 10) y un tratamiento de categorias poco frecuentes. '
    'Mantendre los mismos 35.000 pedidos, la semilla, los predictores y los '
    '24.500/5.250/5.250 registros de entrenamiento/validacion/prueba. '
    f'La linea base a superar es F1={medidas["f1"]:.3f}, '
    f'Recall={medidas["recall"]:.3f}, AP={medidas["ap"]:.3f} y '
    f'costo ilustrativo de R$ {costo_fp_fn:,}. '
    'Solo entrenare con train y evaluare alternativas sobre valid; '
    'no usaremos el test antes del cierre de la semana 16.'
)
respuestas = [respuesta_1, respuesta_2, respuesta_3]
texto_interpretacion = (
    f'# Interpretacion de resultados | S12 | ID {identificador}\n\n'
    '**Borrador automatizado basado en la ejecucion:** revisar, reformular y '
    'sustentar personalmente antes de entregar.\n\n'
    + '\n\n'.join(f'## Pregunta {i}\n\n{texto}'
                        for i, texto in enumerate(respuestas, start=1))
    + '\n\n## Limitaciones\n\n'
    'La muestra solo incluye pedidos entregados; los costos de R$ 8 y R$ 35 '
    'son supuestos didacticos, no cifras oficiales de Olist. '
    'La validacion aleatoria no demuestra desempeño prospectivo. '
    'Las 28 variables excluyen informacion futura de despacho y entrega.\n'
)
# Reemplaza los escapes de salto usados para legibilidad del codigo.
ruta_analisis = carpeta_reporte / f'S12_analisis_ID_{identificador}.md'
ruta_analisis.write_text(texto_interpretacion, encoding='utf-8')

print('JSON individual:', ruta)
print('Interpretacion calculada:', ruta_analisis)
print('Tablas de segmentos: 2 CSV | Figuras: reports/figuras/')
display(Markdown(texto_interpretacion))

# Colab permite descargar el JSON; el notebook y los demas reportes se
# deben subir despues a GitHub porque Colab usa almacenamiento temporal.
try:
    from google.colab import files
    files.download(str(ruta))
except ImportError:
    print('Fuera de Colab: los archivos permanecen en reports/.')



---

### Notas de reproducibilidad y ética

- Los archivos CSV permanecen fuera del repositorio; se descargan desde la revisión de GitHub fijada por el profesor.
- La variable `is_late` se deriva de la fecha real de entrega, pero esa fecha **no** se usa como predictor.
- El notebook no utiliza `test` para entrenar, escoger ponderaciones ni escribir conclusiones S12.
- Los costos representan **supuestos ilustrativos**, no las cuentas operativas de Olist.
- El reporte automatico de interpretación es un **borrador**: se deben revisar las cifras y redactar personalmente las conclusiones para su sustentacion.
